In [0]:
import faker
from rand_engine.core.distinct_core import DistinctCore
from rand_engine.core.numeric_core import NumericCore
from rand_engine.core.datetime_core import DatetimeCore
from rand_engine.core.distinct_utils import DistinctUtils
from pandas import DataFrame as PandasDF
import numpy as np
import pandas as pd
from datetime import datetime as dt, timedelta
from rand_engine.core.distinct_core import DistinctCore
from rand_engine.core.numeric_core import NumericCore
from rand_engine.core.datetime_core import DatetimeCore

from rand_engine.core.distinct_utils import DistinctUtils
from rand_engine.main.dataframe_builder import BulkRandEngine

from datetime import datetime as dt, timedelta

import faker
import csv
import os
import boto3
import logging



class FakeCustomer:

    def __init__(self):
        self.faker = faker.Faker(locale="pt_BR")

    def metadata(self):
        return {
        "user_id": {
            "method": NumericCore.gen_ints_zfilled,
            "parms": dict(length=14)
        },
        "user_type": {     
            "method": DistinctCore.gen_distincts_untyped,
            "parms": dict(distinct=DistinctUtils.handle_distincts_lvl_1({"standard": 80,"premium":15, "gold": 5, None: 7}, 1))
        },
        "first_name": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[self.faker.first_name() for _ in range(1000)])
        },
        "last_name": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[f"{self.faker.last_name()} {self.faker.last_name()}" for _ in range(10000)])
        },
        "income": {
            "method": NumericCore.gen_floats_normal,
            "parms": dict(mean=10000, std=3000, round=2)
        },
        "balance": {
            "method": NumericCore.gen_floats_normal,
            "parms": dict(mean=5000, std=3000, round=2)
        },
        "profession": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[self.faker.job() for _ in range(100)])
        },
        "birth_date": dict(
            method=DatetimeCore.gen_datetimes, 
            parms=dict(start='1971-07-05', end='2013-07-06', format_in="%Y-%m-%d", format_out="%d/%m/%Y")
        ),
        "signup_date": dict(
            method=DatetimeCore.gen_timestamps,
            parms=dict(start="01-01-2021", end="31-12-2025", format="%d-%m-%Y")
        )
    }

    def transformer(self, **kwargs):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            df["income"] = np.where(df["income"] < 0, 0, df["income"])
            for k, v in kwargs.items(): df[k] = v
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            return df
        return wrapped_transformer

    def transformer_cdc_update(self, null_rate, **kwargs):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            for col in df.columns:
                df[col] = np.where(np.random.random(df.shape[0]) < null_rate, None, df[col])
            for k, v in kwargs.items(): df[k] = v
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            return df
        return wrapped_transformer
    
    def debugger(self):
        
        
        data = {
            "id": [1, 2, 3],
            "name": ["marco", "gisele", "tauan"],
            "age": [34, 34, 2]
        }
        df = pd.DataFrame(data)
        return df
    

class FakeOrders:

    def __init__(self):
        self.faker = faker.Faker(locale="pt_BR")

    def metadata(self):
        return {
            "order_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=16)
            },
            "user_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=10)
            },
            "product_id": {
                "method": NumericCore.gen_ints_zfilled,
                "parms": dict(length=3)
            },
            "user_type": {     
            "method": DistinctCore.gen_distincts_untyped,
            "parms": dict(distinct=DistinctUtils.handle_distincts_lvl_1({"standard": 80,"premium":15, "gold": 5, None: 7}, 1))
            },
            "device": {
                "method": DistinctCore.gen_distincts_typed,
                "parms": dict(distinct=["IOS", "Android", "Desktop"])
            },
            "traffic_source": {
                "method": DistinctCore.gen_distincts_typed,
                "parms": dict(distinct=["website", "linkedin", "email"])
            }
        }

    def transformer(self):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            start_time = dt.now() - timedelta(minutes=10)
            end_time = dt.now()
            random_times = pd.to_datetime(np.random.uniform(start_time.timestamp(), end_time.timestamp(), size=len(df)), unit='s').strftime('%Y-%m-%dT%H:%M:%S')
            df["created_at"] = random_times
            return df
        return wrapped_transformer
    

    def debugger(self):
        data = {
            "id": [1, 2, 3],
            "name": ["marco", "gisele", "tauan"],
            "age": [34, 34, 2]
        }
        df = pd.DataFrame(data)
        return df


In [0]:


class LogsGenerator:

  def __init__(self, logger, rand_engine):
    self.logger = logger
    self.rand_engine = rand_engine
    self.s3 = None
    self.bucket = None

  def __metadata_case_web_log_server(self, formato, dt_start, dt_end):
    # Esse método retorna um dicionário com os metadados necessários para gerar os dados usando a biblioteca rand-engine
    # Cada item do dicionário está relacionado a um campo, com exceção da chame "campos_correlacionados_proporcionais" que cria 2 campos correlacionados
    # Ao explorar as possíveis configurações desse dicionário é possível entender quais são as configurações possíveis. Mais 
    fake = faker.Faker(locale="pt_BR")
    metadata = {
      "ip_address":dict(method=DistinctCore.gen_distincts_typed, parms=dict(distinct=[fake.ipv4_public() for i in range(1000)])),
      "identificador": dict(method=DistinctCore.gen_distincts_typed, parms=dict(distinct=["-"])),
      "user": dict(method=DistinctCore.gen_distincts_typed, parms=dict(distinct=["-"])),
      "user_named": dict(method=DistinctCore.gen_distincts_typed, parms=dict(distinct=[fake.first_name().lower().replace(" ", "_") for i in range(1000)])),
      "datetime": dict(
        method=DatetimeCore.gen_datetimes, 
        parms=dict(start=dt_start, end=dt_end, format_in=formato, format_out="%d/%b/%Y:%H:%M:%S")
      ),
      "http_version": dict(
        method=DistinctCore.gen_distincts_typed,
        parms=dict(distinct=DistinctUtils.handle_distincts_lvl_1({"HTTP/1.1": 7, "HTTP/1.0": 3}, 1))
      ),
      "object_size": dict(method=NumericCore.gen_ints, parms=dict(min=0, max=10000)),
      "campos_correlacionados_proporcionais": dict(
        method=       DistinctCore.gen_distincts_typed,
        splitable=    True,
        cols=         ["http_request", "http_status"],
        sep=          ";",
        parms=        dict(distinct=DistinctUtils.handle_distincts_lvl_3({
                          "GET /home": [("200", 7),("400", 2), ("500", 1)],
                          "GET /login": [("200", 5),("400", 3), ("500", 1)],
                          "POST /login": [("201", 4),("404", 2), ("500", 1)],
                          "GET /logout": [("200", 3),("400", 1), ("400", 1)],
                          "POST /signin": [("201", 4),("404", 2), ("500", 1)],
                          "GET /balance": [("200", 3),("400", 1), ("500", 1)],
                          "POST /loans/make_loan": [("200", 3),("400", 1), ("500", 1)],
                          "GET /credit/statement.pdf": [("200", 3),("400", 1), ("500", 1)],
                          "GET /account/statement.pdf": [("200", 3),("400", 1), ("500", 1)],
                          
          }))
      )
    }
    return metadata
  
  def web_server_log_transformer(self, df): 
    # "This method receives the pandas dataframe generated by rand-engine and transform it."
    # Transformation 1: Decides when to give a name to the user based on authenticated endpoints;
    # Transformation 2: Transforms the resulted pandas dataframe into a series in the format of WEB_SERVER_LOGS
    authenticated_endpoints = ['GET /home', 'GET /logout', 'GET /balance', 'GET /credit/statement.pdf', 'GET /account/statement.pdf']
    associate_user_with_http_request = lambda x: x['user_named'] if x['http_request'] in authenticated_endpoints else '-'
    df['user'] = df.apply(associate_user_with_http_request, axis=1)
    df = df['ip_address'] + ' ' + df['identificador'] + ' ' + df['user'] + ' [' + df['datetime'] + ' -0300] "' + \
                        df['http_request'] + ' ' + df['http_version'] + '" ' + df['http_status'] + ' ' + df['object_size'].astype(str)
    return df
  

  def write_data_in_micro_batchs(self, size, metadata, paths):
    max_size = 5*10**4
    n_parts = size // max_size
    micro_batch_size = size // n_parts
    for i in range(n_parts):
      df_pandas = self.rand_engine.create_pandas_df(metadata=metadata, size=micro_batch_size)
      df_pandas = df_pandas.sort_values(by='datetime')
      series_pandas_rand = self.web_server_log_transformer(df_pandas)
      series_pandas_rand.to_csv(paths["local"], sep=' ', index=False, header=False, quoting=csv.QUOTE_NONE, escapechar=' ', mode='a')
    self.logger.info("Local Files Created")
    self.s3.upload_file(paths['local'], self.bucket, paths['s3'])
    self.logger.info(f"Size of local file: {os.path.getsize(paths['local'])}")
    self.logger.info("Files Uploaded to S3")